In [5]:
import pandas as pd

df = pd.read_csv("E:/Projects/BTP/Data/Clean/merged_clean-first.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

print("\nLast 5 rows:")
print(df.tail())

Shape: (2827, 6)

Columns:
['Date', 'Governorate', 'Cases', 'Deaths', 'New_Cases', 'New_Cases_flag']

First 5 rows:
         Date Governorate  Cases  Deaths  New_Cases New_Cases_flag
0  2017-05-22       Abyan   1068      10     1068.0             ok
1  2017-05-27       Abyan   1365      15      297.0             ok
2  2017-05-30       Abyan   1587      16      222.0             ok
3  2017-06-04       Abyan   1920      16      333.0             ok
4  2017-06-07       Abyan   2171      17      251.0             ok

Last 5 rows:
            Date Governorate  Cases  Deaths  New_Cases New_Cases_flag
2822  2018-01-21       Taizz  63050     187      679.0             ok
2823  2018-01-28       Taizz  63282     187      232.0             ok
2824  2018-02-04       Taizz  63457     188      175.0             ok
2825  2018-02-11       Taizz  63592     188      135.0             ok
2826  2018-02-18       Taizz  63696     188      104.0             ok


In [6]:
print("Missing values per column:")
print(df.isna().sum())

print("\nTotal missing values:")
print(df.isna().sum().sum())

Missing values per column:
Date              0
Governorate       0
Cases             0
Deaths            0
New_Cases         0
New_Cases_flag    0
dtype: int64

Total missing values:
0


In [7]:
print(df.dtypes)

Date                  str
Governorate           str
Cases               int64
Deaths              int64
New_Cases         float64
New_Cases_flag        str
dtype: object


In [8]:
print("\nData types:")
print(df.dtypes)

print("\nSample values:")
print(df.head())


Data types:
Date                  str
Governorate           str
Cases               int64
Deaths              int64
New_Cases         float64
New_Cases_flag        str
dtype: object

Sample values:
         Date Governorate  Cases  Deaths  New_Cases New_Cases_flag
0  2017-05-22       Abyan   1068      10     1068.0             ok
1  2017-05-27       Abyan   1365      15      297.0             ok
2  2017-05-30       Abyan   1587      16      222.0             ok
3  2017-06-04       Abyan   1920      16      333.0             ok
4  2017-06-07       Abyan   2171      17      251.0             ok


In [9]:
converted_dates = pd.to_datetime(df["Date"], errors="coerce")

print("Invalid dates:", converted_dates.isna().sum())

print("\nDate range:")
print("Earliest:", converted_dates.min())
print("Latest:", converted_dates.max())

Invalid dates: 0

Date range:
Earliest: 2017-05-22 00:00:00
Latest: 2018-02-18 00:00:00


In [10]:
print("Number of unique governorates:")
print(df["Governorate"].nunique())

print("\nAll unique governorate names:")
print(sorted(df["Governorate"].unique()))

Number of unique governorates:
21

All unique governorate names:
['Abyan', 'Aden', 'Al Bayda', "Al Dhale'e", 'Al Hudaydah', 'Al Jawf', 'Al Maharah', 'Al Mahwit', 'Amanat Al Asimah', 'Amran', 'Dhamar', 'Hadramawt', 'Hajjah', 'Ibb', 'Lahj', 'Marib', 'Raymah', "Sa'ada", "Sana'a", 'Shabwah', 'Taizz']


In [11]:
hadramawt = df[df["Governorate"] == "Hadramawt"]

print("Number of Hadramawt rows:")
print(len(hadramawt))

print("\nFirst 10 Hadramawt rows:")
print(hadramawt.head(10))

print("\nLast 10 Hadramawt rows:")
print(hadramawt.tail(10))

Number of Hadramawt rows:
114

First 10 Hadramawt rows:
            Date Governorate  Cases  Deaths  New_Cases      New_Cases_flag
1491  2017-06-30   Hadramawt      2       0        2.0                  ok
1492  2017-07-01   Hadramawt      2       0        0.0                  ok
1493  2017-07-02   Hadramawt      2       0        0.0                  ok
1494  2017-07-03   Hadramawt      2       0        0.0                  ok
1495  2017-07-04   Hadramawt      2       0        0.0                  ok
1496  2017-07-05   Hadramawt      2       0        0.0                  ok
1497  2017-07-06   Hadramawt      6       0        4.0                  ok
1498  2017-07-07   Hadramawt      6       0        0.0                  ok
1499  2017-07-10   Hadramawt      4       0        0.0  corrected_negative
1500  2017-07-11   Hadramawt      4       0        0.0                  ok

Last 10 Hadramawt rows:
            Date Governorate  Cases  Deaths  New_Cases New_Cases_flag
1595  2017-11-26   Hadra

In [12]:
duplicates = df[df.duplicated(
    subset=["Date", "Governorate"],
    keep=False
)]

print("Number of duplicate rows:")
print(len(duplicates))

print("\nNumber of duplicate Date + Governorate combinations:")
print(
    df.duplicated(
        subset=["Date", "Governorate"]
    ).sum()
)

print("\nDuplicate records:")
print(duplicates.sort_values(["Governorate", "Date"]))

Number of duplicate rows:
0

Number of duplicate Date + Governorate combinations:
0

Duplicate records:
Empty DataFrame
Columns: [Date, Governorate, Cases, Deaths, New_Cases, New_Cases_flag]
Index: []


In [13]:
# Work on a copy so we don't modify your cleaned dataset
audit = df.copy()

# Convert Date only for sorting/comparison
audit["Date_dt"] = pd.to_datetime(audit["Date"])

# Sort chronologically within each governorate
audit = audit.sort_values(["Governorate", "Date_dt"])

# Calculate the raw difference in cumulative Cases
audit["Expected_New_Cases"] = (
    audit.groupby("Governorate")["Cases"].diff()
)

# First observation of each governorate:
# expected New_Cases = cumulative Cases
audit["Expected_New_Cases"] = audit["Expected_New_Cases"].fillna(
    audit["Cases"]
)

# Negative differences were intentionally clipped to 0
audit["Expected_New_Cases"] = audit["Expected_New_Cases"].clip(lower=0)

# Find mismatches
mismatches = audit[
    audit["New_Cases"] != audit["Expected_New_Cases"]
]

print("Total rows:", len(audit))
print("Rows where New_Cases does not match expected value:", len(mismatches))

print("\nMismatches:")
print(
    mismatches[
        [
            "Date",
            "Governorate",
            "Cases",
            "New_Cases",
            "New_Cases_flag",
            "Expected_New_Cases"
        ]
    ].head(20)
)


Total rows: 2827
Rows where New_Cases does not match expected value: 0

Mismatches:
Empty DataFrame
Columns: [Date, Governorate, Cases, New_Cases, New_Cases_flag, Expected_New_Cases]
Index: []


In [14]:
print("Negative New_Cases remaining:")
print((df["New_Cases"] < 0).sum())

print("\nNew_Cases_flag counts:")
print(df["New_Cases_flag"].value_counts())

print("\nCorrected-negative rows:")
corrected = df[df["New_Cases_flag"] == "corrected_negative"]

print(corrected[
    ["Date", "Governorate", "Cases", "New_Cases", "New_Cases_flag"]
].to_string(index=False))

Negative New_Cases remaining:
0

New_Cases_flag counts:
New_Cases_flag
ok                    2825
corrected_negative       2
Name: count, dtype: int64

Corrected-negative rows:
      Date Governorate  Cases  New_Cases     New_Cases_flag
2017-07-10   Hadramawt      4        0.0 corrected_negative
2017-10-10   Hadramawt    559        0.0 corrected_negative


In [15]:
# Convert Date for chronological sorting
audit_first = df.copy()
audit_first["Date_dt"] = pd.to_datetime(audit_first["Date"])

# Sort by governorate and date
audit_first = audit_first.sort_values(
    ["Governorate", "Date_dt"]
)

# Get the first chronological row of every governorate
first_rows = audit_first.groupby(
    "Governorate",
    as_index=False
).first()

# Check whether New_Cases == Cases
first_rows["matches"] = (
    first_rows["New_Cases"] == first_rows["Cases"]
)

print("First observation of each governorate:")
print(
    first_rows[
        [
            "Governorate",
            "Date",
            "Cases",
            "New_Cases",
            "matches"
        ]
    ].to_string(index=False)
)

print("\nNumber of governorates:")
print(len(first_rows))

print("\nNumber where New_Cases == Cases:")
print(first_rows["matches"].sum())

print("\nNumber of mismatches:")
print((~first_rows["matches"]).sum())

First observation of each governorate:
     Governorate       Date  Cases  New_Cases  matches
           Abyan 2017-05-22   1068     1068.0     True
            Aden 2017-05-22    489      489.0     True
        Al Bayda 2017-05-22   1498     1498.0     True
      Al Dhale'e 2017-05-22   1401     1401.0     True
     Al Hudaydah 2017-05-22   1397     1397.0     True
         Al Jawf 2017-05-22    189      189.0     True
      Al Maharah 2017-06-10     46       46.0     True
       Al Mahwit 2017-05-22   2486     2486.0     True
Amanat Al Asimah 2017-05-22   9216     9216.0     True
           Amran 2017-05-22   3743     3743.0     True
          Dhamar 2017-05-22   1617     1617.0     True
       Hadramawt 2017-06-30      2        2.0     True
          Hajjah 2017-05-22   4664     4664.0     True
             Ibb 2017-05-22   1378     1378.0     True
            Lahj 2017-05-22    168      168.0     True
           Marib 2017-05-22      2        2.0     True
          Raymah 2017-05-2

In [16]:
print("Numerical statistics:")
print(df[["Cases", "Deaths", "New_Cases"]].describe())

Numerical statistics:
               Cases       Deaths    New_Cases
count    2827.000000  2827.000000  2827.000000
mean    26869.440042    89.812876   376.296427
std     28299.864531    96.261266   574.304185
min         2.000000     0.000000     0.000000
25%      4105.500000    12.000000    32.000000
50%     17585.000000    60.000000   199.000000
75%     41403.000000   141.000000   503.000000
max    155908.000000   422.000000  9216.000000


In [17]:
print("\nMinimum values:")
print(df[["Cases", "Deaths", "New_Cases"]].min())

print("\nMaximum values:")
print(df[["Cases", "Deaths", "New_Cases"]].max())


Minimum values:
Cases        2.0
Deaths       0.0
New_Cases    0.0
dtype: float64

Maximum values:
Cases        155908.0
Deaths          422.0
New_Cases      9216.0
dtype: float64


In [18]:
print("Number of observations per governorate:")
print(
    df["Governorate"]
    .value_counts()
    .sort_index()
)

Number of observations per governorate:
Governorate
Abyan               136
Aden                136
Al Bayda            136
Al Dhale'e          136
Al Hudaydah         136
Al Jawf             136
Al Maharah          131
Al Mahwit           136
Amanat Al Asimah    136
Amran               136
Dhamar              136
Hadramawt           114
Hajjah              136
Ibb                 136
Lahj                136
Marib               136
Raymah              136
Sa'ada              135
Sana'a              136
Shabwah             135
Taizz               136
Name: count, dtype: int64


In [19]:
frequency = df.copy()

frequency["Date"] = pd.to_datetime(frequency["Date"])

frequency = frequency.sort_values(
    ["Governorate", "Date"]
)

frequency["Days_Since_Previous"] = (
    frequency.groupby("Governorate")["Date"]
    .diff()
    .dt.days
)

print("\nDate difference statistics:")
print(
    frequency["Days_Since_Previous"]
    .describe()
)


Date difference statistics:
count    2806.000000
mean        2.011404
std         2.661780
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        21.000000
Name: Days_Since_Previous, dtype: float64
